<a href="https://colab.research.google.com/github/wjdwogns2873-web/deep-learning-study/blob/main/Kaggle_Study_Practice/09_Rice_Image_Practice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("muratkokludataset/rice-image-dataset")

print("Path to dataset files:", path)

In [ ]:
# !ls -al /root/.cache/kagglehub/datasets/muratkokludataset/rice-image-dataset/versions/1/Rice_Image_Dataset
!ls -al /kaggle/input/rice-image-dataset/Rice_Image_Dataset/

In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold

N_SPLITS = 5
# DATA_DIR = '/root/.cache/kagglehub/datasets/muratkokludataset/rice-image-dataset/versions/1/Rice_Image_Dataset'
DATA_DIR = '/kaggle/input/rice-image-dataset/Rice_Image_Dataset/'
categories = ['Arborio', 'Basmati', 'Ipsala', 'Jasmine', 'Karacadag']

data = []
for category in categories:
    folder_path = os.path.join(DATA_DIR, category)
    for img_name in os.listdir(folder_path):
        if img_name.endswith(('.jpg', '.jpeg', '.png')):
            data.append({
                'image_path': os.path.join(folder_path, img_name),
                'label_name': category
            })

df = pd.DataFrame(data)

label_map = {name: i for i, name in enumerate(categories)}
df['label'] = df['label_name'].map(label_map)

train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)

train_val_df = train_val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_val_df['fold'] = -1
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

for fold, (train_index, val_idx) in enumerate(skf.split(train_val_df, train_val_df['label'])):
    train_val_df.loc[val_idx, 'fold'] = fold

print(f"Train/Val 데이터 개수: {len(train_val_df)}, Test 데이터 개수: {len(test_df)}")

```
# 1
train_val_df, test_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['label']
)

random_state=42: 난수 생성 시드값입니다. 이 숫자를 고정해 두어야 코드를 여러 번 실행해도
항상 똑같은 데이터들이 Test 세트로 분리됩니다.(실험의 재현성을 위해 필수입니다.)

stratify=df['fold']: (매우 중요) 운이 안 좋아서 분리된 Test 세트에 특정 품종만 몰려 있으면 안됩니다.
원본 데이터의 품종별 비율(예: A:20%, B:20%..)을 그대로 유지하면서 15%로 쪼개라는 의미입니다.

train_val_df = train_val_df.reset_index(drop=True)

# 2
reset_index(drop=True)를 하는 이유

자바에서 List에서 데이터 몇 개를 remove()하고 나면 인덱스가 중간중간 비거나 어긋나지 않지만, Pandas DataFrame은 그렇지 않습니다.
75000개의 행 중에서 15%가 무작위로 빠져나가면 남은 데이터들의 인덱스가 [0, 1, 3, 4, 11...]처럼 중간중간 비게 됩니다.

reset_index(): 이 구멍난 인덱스를 다시 [0, 1, 2, 3, 4, 5..]처럼 예쁘게 재정렬해 줍니다.
drop=True: 인덱스를 새로 만들면 기존에 구멍 나 있던 구 인덱스 번호가 index라는 새로운 컬럼으로 튀어나옵니다.
필요없는 데이터이므로, 구 인덱스를 과감히 버려라(drop)는 뜻입니다.

# 3
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

데이터를 5개의 폴드(0~4)로 나누겠다는 분할기(Splitter) 객체를 생성한 것입니다.

# 4
for fold, (train_index, val_idx) in enumerate(skf.split(train_val_df, train_val_df['label'])):
    train_val_df.loc[val_idx, 'fold'] = fold

(train_index, val_idx) 안에는 단순한 정수(index)들이 가득 담긴 파이썬 리스트(배열)가 들어있습니다.
예를 들어 train_val_df 데이터가 총 10개 있다고 가정합니다. [0, 1, ..., 8, 9]
group A[2, 5] group B[0, 7] group C[4, 9]...

0번째 루프일 때.
val_idx = [2, 5] group A
train_index = [..] group A를 제외한 나머지 전부

1번째 루프일 때.
val_idx = [0, 7] group B
train_index = [..] group B를 제외한 나머지 전부
```

In [ ]:
import cv2
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2

class RiceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row['image_path'])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = int(row['label'])

        if self.transform:
            # 1. 밖에서 정의한 설정을 주입받아 실행
            augmented = self.transform(image=image)
            # 2. 그 결과물(딕셔너리)에서 진짜 '이미지 행렬'만 쏙 빼내기
            image = augmented['image']

        return image, torch.tensor(label, dtype=torch.long)

train_transform = A.Compose([
    A.Resize(128, 128),
    A.HorizontalFlip(p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(128, 128),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

```
# 1
cv2.imread() vs Image.open()
두 메서드 모두 "디스크에 있는 이미지 파일 경로를 받아 메모리로 읽어오는 기능"

Image.open(): 파이썬 전통적인 이미지 처리 라이브러리입니다. 이미지를 자체적인 PIL Image 객체로 반환합니다.

cv2.imread(): C++ 기반으로 만들어져 빠르고 강력한 컴퓨터 비전 전용 라이브러리입니다.
이미지를 읽는 순간 NumPy Array 형태로 곧바로 반환해 줍니다.

대용량 데이터셋을 다룰 때는 연산 속도가 중요하므로 Image.open() 보다 속도가 빠른 cv2.imread()를 많이 씁니다.

# 2
cv2.cvtColor(image, cv2.COLOR_BGR2RGB)를 하는 이유

일반적인 모든 이미지 파일과 딥러닝 모델은 색상을 RGB 순서로 인식합니다.
하지만 옛날에 만들어진 OpenCV는 BGR 순서로 읽어옵니다. 그래서 이것을 RGB로 보정하는 겁니다.

# 3
augmented = self.transform(image=image)
Albumentations 라이브러리의 특징인데, key=value 구조의 딕셔너리 형태로 입출력을 받습니다.
그래서 변환 결과물인 augmented 안에는 {'image': 변환된이미지행렬} 구조가 리턴됩니다.

image = augmented['image']
변환 완료된 이미지 행렬 데이터만 쏙 빼내서 image 변수에 다시 대입해 주는 것입니다.

# 4
torch.tensor(label, dtype=torch.long)를 하는 이유
파이토치로 손실 함수를 계산할 때, 정답지는 반드시 64비트 부호 있는 정수형(torch.Long)이어야만
엔진이 에러를 내지 않습니다.
만약 int나 float 상태로 리턴하면 타입이 맞지 않다고 에러를 내뱉습니다.
```

In [ ]:
from torch.utils.data import DataLoader
import timm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
epochs = 3
BATCH_SIZE = 128

test_dataset = RiceDataset(test_df, transform=val_test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
total_test_preds = torch.zeros(len(test_df), len(categories)).to(device)

for fold in range(N_SPLITS):
    train_data = train_val_df[train_val_df['fold'] != fold].reset_index(drop=True)
    val_data = train_val_df[train_val_df['fold'] == fold].reset_index(drop=True)

    train_loader = DataLoader(RiceDataset(train_data, transform=train_transform), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    val_loader = DataLoader(RiceDataset(val_data, transform=val_test_transform), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

    model = timm.create_model('resnet18', pretrained=True, num_classes=5).to(device)
    criterion = torch.nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

    best_val_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_correct = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * labels.size(0)

            preds = outputs.argmax(1)
            train_correct += (preds == labels).sum().item()

        epoch_train_loss = train_loss / len(train_data)
        epoch_train_acc = train_correct / len(train_data) * 100
        print(f"Epoch {epoch+1} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.2f}%")

        model.eval()
        val_loss = 0.0
        correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * labels.size(0)
                preds = outputs.argmax(1)

                correct += (preds == labels).sum().item()

        epoch_val_loss = val_loss / len(val_data)
        epoch_val_acc = correct / len(val_data) * 100
        print(f"Epoch {epoch+1} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.2f}%")

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), f"best_model_fold{fold}.pth")

    print(f"--> Load Best Model of Fold {fold} and Predict Test Data")
    best_model = timm.create_model('resnet18', pretrained=False, num_classes=5).to(device)
    best_model.load_state_dict(torch.load(f"best_model_fold{fold}.pth"))
    best_model.eval()

    fold_test_preds = []
    with torch.no_grad():
        for images, _ in test_loader:
            images = images.to(device)
            outputs = best_model(images)

            probs = outputs.softmax(1)
            fold_test_preds.append(probs)

    total_test_preds += torch.cat(fold_test_preds, 0)

final_test_preds = total_test_preds / 5
final_classes = final_test_preds.argmax(1).cpu().numpy()

test_df['predicted_label'] = final_classes
test_df.to_csv('submission.csv', index=False)

```
# 1
before: total_test_preds = torch.zeros((len(test_df), len(categories))).to(device)
after: total_test_preds = torch.zeros(len(test_df), len(categories)).to(device)
괄호를 빼도 된다.

# 2
train_data = train_val_df[train_val_df['fold'] != fold].reset_index(drop=True)
resnet_index(drop=True)를 또 하는 이유

train_val_df['fold'] != fold
위와 같이 필터링을 거치면 또 중간중간 인덱스 누수가 발생하기 때문입니다.
따라서 필터링 후에는 무조건 세트로 인덱스를 리셋해 주는 것이 국룰입니다.

# 3
final_classes = final_test_preds.argmax(1).cpu().numpy()
.argmax()를 하게 되면..
확률 중 가장 큰 녀석의 인덱스 번호 1개만 남기 때문에 열(Column) 차원이 압축되어 사라집니다.
[약 60000, 1]이 아니라 [약 60000]입니다.
```